# Tumor in the network: disconnection, resection and model functional connectivity

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/arvidl/AI-assisted-glioma-research/blob/main/notebooks/11_network_disconnection.ipynb)

*In Google Colab, the first code cell clones the public repository and puts its package on the path; Colab's own NumPy, Matplotlib and ipywidgets are used, not the locked versions. No data are downloaded: the network is constructed in code.*

**Question.** When a glioma infiltrates white matter, and part of it is then resected, which
connections between brain regions are weakened or lost? And how does that change the correlated
activity that a simple network model produces?

**Evidence.** Synthetic. A structural network is constructed in code on the coronal phantom of
notebook 02 and lesioned by that notebook's tissue-guided tumor (its Figure 2). Activity follows a
linear network model whose functional connectivity has an exact solution. Nothing is fitted to a
patient, and no data are read.

**What you will learn.**
- The basic vocabulary of graph theory (nodes, edges, weights, walks, paths, efficiency, spectra),
  and why neuroscience, clinical neuroscience and neuro-oncology adopted it.
- How a lesion is represented in a network: as damage to edges, not only as removed nodes.
- Why two regions can be correlated in a model without a direct connection, and why a lesion can
  change their correlation even when their own connection is intact.
- What a resection does in this construction, and what would be needed before any of it could be
  compared with cognitive outcomes.

**Three reading tracks.** The model, the code and the figures are the same for everyone. Most
sections then have one paragraph for each of three kinds of reader. Read the track closest to your
background, or all three.
- *Cognitive neuroscience track.* For readers who study how cognitive functions arise from
  distributed brain systems. It follows the disconnection idea from classic syndromes to network
  models, asks which claims about function the model can and cannot make, and is explicit about
  what a single coronal slice cannot represent: frontal cortex, and long association tracts that
  leave the plane. It assumes familiarity with resting-state functional connectivity, not with
  graph theory.
- *Neuro-oncology and clinical track.* For clinicians and clinical researchers. It reads each figure
  as a question about infiltration, white-matter tracts and extent of resection, relates it to awake
  mapping, tractography-guided surgery and postoperative cognition, and says why nothing here is a
  surgical or prognostic result. It assumes clinical familiarity with glioma, not with modelling.
- *Data-analysis and modelling track.* For readers who build or audit models. It states every rule,
  the exact solution and its stability condition, and the numerical controls. It also lists what
  would have to be replaced (nodes, edges, damage rule, dynamics, cognitive mapping) before the same
  chain could be run on patient data. It assumes linear algebra; section 1 introduces the graph
  theory.

**Map of the notebook.** Section 1 introduces graph theory and its use in neuroscience. Section 2
builds the phantom network. Section 3 puts a linear model of activity on it and shows how structure
becomes functional connectivity. Section 4 lesions the network with the growing tumor (Figure 1),
and section 5 adds a constructed resection and asks what a cognitive comparison would need
(Figure 2). Section 6 says what carries over to one patient's connectome in notebook 12. Section 7
has exercises for each track, and section 8 the references.

Scientific functions live in `glioma_ai.network` and `glioma_ai.network_figures`; this notebook
exposes their assumptions and interprets their outputs. Locally, run on CPU with the locked
project environment documented in README.md. The explorer in section 5 needs a running kernel with
the `notebooks` dependency group; the static figures are produced either way.

## Motivation: from a growing tumor to a disconnected network

Notebook 02 ends with a tumor that starts in left parietal white matter and, guided by tissue type
and fibre direction, reaches the other hemisphere through the corpus callosum. A tumor like this
does not only occupy tissue. It lies across fibres that connect regions far from it. This notebook
builds the smallest version of that chain that can still be checked line by line:

tissue density → tract damage → structural weights → model activity → functional connectivity.

It then asks what a comparison with cognitive scores (notebook 10) would require. Each step is a
stated rule, so every effect in the figures can be traced back to its cause.

**Cognitive neuroscience track.** A long tradition explains deficits by the connections a lesion
interrupts, not only by the cortex it destroys. [Catani and ffytche (2005)](https://doi.org/10.1093/brain/awh622) reviewed the
rise, fall and return of such *disconnection syndromes*. Network models make the idea quantitative.
In models built on macaque cortical anatomy, [Honey and Sporns (2008)](https://doi.org/10.1002/hbm.20579) found that lesion
effects extend beyond the immediate neighbours of the damaged region.
[Alstott et al. (2009)](https://doi.org/10.1371/journal.pcbi.1000408) removed regions from a human structural connectome and simulated
how activity elsewhere changed. [Honey et al. (2009)](https://doi.org/10.1073/pnas.0811168106) showed that structural
connectivity partly predicts resting-state functional connectivity, including strong functional
links between regions with no direct structural connection. In 114 stroke patients,
[Griffis et al. (2019)](https://doi.org/10.1016/j.celrep.2019.07.100) found that measures of structural disconnection explained the
disruption of functional connectivity better than measures of focal cortical damage, and that
interhemispheric disconnection was linked to deficits in several behavioural domains. Glioma is not
stroke: it grows slowly, infiltrates rather than destroys, and the brain adapts while it grows.
Recent reviews cover connectome findings in glioma ([Salvalaggio et al. 2025](https://doi.org/10.1097/CCO.0000000000001190)), glioma's
impact on cognition from macrostructure to microstructure ([Cayuela et al. 2025](https://doi.org/10.1093/noajnl/vdaf003)), and
network accounts from tumor biology through to symptoms ([Douw et al. 2026](https://doi.org/10.1038/s41582-025-01171-x)). One limit of
this notebook is visible at once: the slice has no frontal lobe, so a planning score cannot be
placed on it.

**Neuro-oncology and clinical track.** The surgical question is how much infiltrated tissue to
remove, and what that costs the cortex and tracts that remain. From serial awake mappings,
[Duffau (2014)](https://doi.org/10.1016/j.cortex.2013.08.005) argued that cortex can reorganise extensively but that white-matter
connectivity is a main limit of that plasticity, which would explain the lack of recovery after
extensive subcortical damage. [Herbet (2024)](https://doi.org/10.1093/brain/awae093) reviews predictors of cognition after
glioma surgery, among them disconnection of white-matter pathways ("connectotomy") and plasticity.
A 2026 systematic review associates connectome- and tractography-guided resection with tract
preservation, postoperative cognition and extent of resection
([Tykocki and Rakasz 2026](https://doi.org/10.1007/s10143-025-04081-0)). Naming one tract and grading it intact, partly cut or cut is
the clinical counterpart of the tract integrity used below. [Nasiri et al. (2026)](https://doi.org/10.1016/j.bas.2026.106254) did
that for the left frontal aslant tract and the arcuate fasciculus in 23 patients. After awake
resection of language-eloquent glioma, early changes in the functional connectome have been
associated with needing more help in daily activities ([Noll et al. 2025](https://doi.org/10.1093/noajnl/vdaf192)). In motor-area
gliomas, [Wang et al. (2026)](https://doi.org/10.3171/2026.4.JNS26181) used resting-state fMRI, diffusion MRI and callosal
tractography to predict motor outcome. This notebook answers none of those questions. Its resection
is a threshold on a synthetic density, and it has no plasticity, oedema or recovery. What it offers
is a setting, with every rule visible, in which to see why a resection that spares cortex can still
disconnect it.

**Data-analysis and modelling track.** [Aerts et al. (2016)](https://doi.org/10.1093/brain/aww194) reviewed computational and
empirical lesion studies of brain networks, including brain tumors, and concluded that lesion
effects depend critically on where in the network the damage lies.
[Aerts et al. (2020)](https://doi.org/10.1016/j.neuroimage.2020.116738) modelled dynamics before and after resection with The Virtual
Brain, in the open cohort described by [Aerts et al. (2022)](https://doi.org/10.1038/s41597-022-01806-4), the same study family as
notebooks 10 and 12. The model here is a small, exactly solvable limit of that idea: a linear,
noise-driven system on a network built in code, whose stationary covariance solves a Lyapunov
equation. Linear models are not only toys. [Abdelnour et al. (2014)](https://doi.org/10.1016/j.neuroimage.2013.12.039) reported that a
linear graph-diffusion model, a close relative of the one used here, captured the relation between
structural and resting-state functional connectivity better than earlier nonlinear approaches.
Exactness matters for teaching: every number below is either a rule or the solution of an
equation, so there is no fitting and no sampling noise to hide behind. Section 6 says what
notebook 12 keeps of this chain when the network is no longer a phantom. Neither notebook fits a
cognitive score. A 2026 commentary argues that a glioma connectivity map should not be used for a
single patient until the clinical state of the scan is recorded and the map's added value is
validated ([Kong and Wei 2026](https://doi.org/10.1002/nbm.70391)).

## AI contribution and verification

**Question.** How does a lesion that grows along white matter and crosses the corpus callosum
change a network's structure and its modelled correlated activity, before and after a resection?

**AI contribution.**
- *Trained imaging AI*: none.
- *Generative research assistance* (development only; records in `docs/ai-use/`): on
  10 October 2026 a Cursor agent (model and settings not captured) wrote the network
  construction, damage rule, linear model and its controls, figures, explorer, tests and the first
  text ([record](../docs/ai-use/2026-10-10-notebook-14-network.md); the notebook was then numbered
  14). On 14 October 2026 Antigravity added the Colab setup
  ([record](../docs/ai-use/2026-10-14-publish-notebooks-11-12.md)). Later that day a Cursor agent
  (model and settings not captured) added the three reading tracks, the glossary, the exercises,
  the reference list, the introduction to notebook 12 and further 2025–2026 references
  ([record](../docs/ai-use/2026-10-14-notebook-11-tracks.md)). On 15 October 2026 a Cursor agent
  (reported as Claude Opus 5.5; settings not captured) revised the language, expanded the tracks and
  the glossary, added the graph-theory primer of section 1 and the walk decomposition of section 3
  (`walk_shares`, with a test), moved the model section before the figures that use it, retitled
  sections 4–6, rewrote the exercises and added references
  ([record](../docs/ai-use/2026-10-15-notebook-11-graph-theory.md)). The model, its parameters and
  the figures were not changed.
- *Conventional code* (rerun every time, no model): path search, weights, damage, exact
  covariance, walk decomposition, simulation and figures. Rebuilding the notebook calls no LLM.

**Investigator choices.** The investigator asked for a notebook that links tumor growth
(notebook 02) to cognition (notebook 10), mirroring the digital-twin questions of the talk
("how is white matter organised and how is structural connectivity disrupted?", "does predicted
disruption match observed cognitive change?"). From options the assistant offered, the
investigator chose a network constructed in code rather than an atlas or patient connectome, a
linear model with an exact solution rather than oscillators, and lesions taken from notebook 02's
Figure 2 density. On 15 October 2026 they asked for clearer language, fuller tracks and glossary,
a more rigorous and didactic treatment of graph theory, better-motivated sections, reconsidered
headings and exercises, and further references. Within that scope the design decisions are:
- *Nodes*: 14 grey-matter regions of the phantom, seven per hemisphere: medial wall above the
  callosum, five angular sectors of the cortical ribbon and the thalamus. Names such as
  "superior parietal" are approximate analogues at a parietal coronal level, not atlas labels.
- *Tracts*: the shortest 8-connected path through white matter between two regions, entering and
  leaving grey matter only at its ends. Because the hemispheres connect only through the corpus
  callosum, every interhemispheric path crosses it. A 2D shortest path is a crude proxy for a
  fibre bundle: it hugs boundaries and ignores routes that leave the slice.
- *Weights*: $w_{ij}=\exp(-L_{ij}/\lambda)$ with path length $L_{ij}$ (mm) and decay length
  $\lambda=40$ mm, a teaching choice similar in form to the exponential distance rule reported for
  macaque tract tracing by [Ercsey-Ravasz et al. (2013)](https://doi.org/10.1016/j.neuron.2013.07.036).
- *Damage rule*: tract integrity
  $s_{ij}=\exp\!\big(-\tfrac{\Delta s}{\ell_d}\sum_{k\in\text{path}} c_k\big)$ with grid spacing
  $\Delta s=1$ mm and damage length $\ell_d=10$ mm, so loss accumulates with the infiltrated
  length. A tract crossing a resection cavity is cut ($s_{ij}=0$). The lesioned weight is
  $s_{ij}w_{ij}$. The density proxy $c$ is not a measured fibre loss; a harsher alternative,
  $s_{ij}=1-\max c$, is available as `rule="max"`.
- *Resection*: left-hemisphere white matter where $c\ge0.3$, sparing the corpus callosum and all
  grey-matter regions, at day 120. Threshold, day and sparing rule are teaching choices, not a
  surgical plan.
- *Dynamics*: $\dot{\mathbf x}=-\mathbf x+gW\mathbf x+\sigma\boldsymbol\xi$ (section 3), with
  $W$ the weights divided by the healthy network's largest eigenvalue and $g=0.8$.
- *Cognitive link*: an assumed, unsourced table from CANTAB measures to phantom regions, used only
  as a template for a comparison; planning (SOC) depends on frontal regions this slice lacks.

**Verification.**
- *Geometry*: every region is non-empty and on its own side; every pair is reachable; no path
  crosses CSF or other grey matter; every interhemispheric path crosses the corpus callosum.
- *Damage rule*: no tumor gives integrity 1 everywhere; a pointwise larger density never gives
  higher integrity (both rules); resection cuts only tracts through the cavity.
- *Stability*: a lesion only removes weight, so it cannot raise the largest eigenvalue
  (Perron–Frobenius), and a stable healthy model stays stable.
- *Exact model functional connectivity*: the covariance solves the Lyapunov equation to rounding
  error. Simulations are checked in two separate parts: the time-step bias, from the exact
  covariance of the discretised scheme, halves when the step halves; the sampling error of a
  simulation shrinks with longer runs.
- *Walk decomposition*: on a three-node chain the shares equal the analytic Neumann series; on the
  phantom they rise monotonically to 1.
- *Not verified here*: any resemblance of the phantom network to a real connectome, any relation
  between $c$ and fibre loss, haemodynamics (the model activity is not BOLD), plasticity and
  recovery after surgery, and any relation to observed cognition.

**Evidence status.** Synthetic throughout. The cited studies motivate the questions; nothing is
fitted to them or reproduces their results. The cognitive table is an assumption awaiting a
sourced replacement. There is no clinical prediction and no measured AI benefit.

## Glossary: acronyms, concepts and definitions for the three tracks

Refer back as needed. Symbols are defined again, with their units, where they are first used.

**Acronyms**

| Acronym | Expansion | Role in this notebook |
| --- | --- | --- |
| BOLD | Blood-oxygen-level-dependent signal | What fMRI records. The model activity here is *not* BOLD. |
| CANTAB | Cambridge Neuropsychological Test Automated Battery | Cognitive battery of notebook 10; Figure 2D. |
| MOT, RTI, RVP, SSP, SOC | Motor screening, reaction time, rapid visual information processing, spatial span, Stockings of Cambridge | The CANTAB measures named in Figure 2D. SOC (planning) has no region on this slice. |
| CSF | Cerebrospinal fluid | Phantom tissue that tract proxies may not cross. |
| dMRI | Diffusion MRI | Source of real tractography (notebook 12); not used here. |
| FC, SC | Functional and structural connectivity | See the concepts table. |
| (rs-)fMRI | (Resting-state) functional MRI | Source of measured FC in patients. |
| MEG | Magnetoencephalography | Used by Bartolomei et al. (2006) for an early graph analysis of brain-tumor networks. |
| TVB | The Virtual Brain | Simulation platform used by Aerts et al. (2020). |

**Concepts.** Each row gives one reading per track. The last column is the definition the code
uses.

| Term | Cognitive neuroscience | Neuro-oncology and clinical | Data analysis and modelling |
| --- | --- | --- | --- |
| Graph, network | The brain as parts and links, so that a function can be attributed to a pattern of links rather than to one part. | A map of which regions are joined by which tracts; a lesion becomes damage to the map. | $G=(V,E)$ with $n$ nodes; here undirected and weighted, stored as a symmetric matrix $W$. |
| Node | A region assumed to act as a unit. | A cortical or deep grey-matter area. Here 14 phantom regions named by analogy, not atlas labels. | Index $i=1,\dots,n$. Network measures depend on how nodes are defined (Zalesky et al. 2010). |
| Edge, weight $w_{ij}$ | A potential route of interaction between two regions. | A white-matter tract proxy; a longer tract has a smaller weight. | $w_{ij}=\exp(-L_{ij}/\lambda)\in(0,1]$, $w_{ij}=w_{ji}$, $w_{ii}=0$. |
| Structural connectivity (SC) | The wiring. | Estimated in patients from dMRI tractography; distorted by oedema, mass effect and surgery. | The matrix $W$ (healthy) or $s_{ij}w_{ij}$ (lesioned). |
| Functional connectivity (FC) | Statistical dependence of regional activity. It can exist without a direct connection. | Measured with rs-fMRI or MEG; a tumor can alter it far from the lesion. | Here the correlation matrix of the linear model's stationary activity, not a measurement. |
| Walk, path | Indirect routes through intermediate regions along which influence may spread. | An indirect route uses a third region and its tracts; cutting any of them affects the route. | A walk is any sequence of edges and may revisit nodes; a path does not. $(W^k)_{ij}$ sums the weight products of all walks of length $k$ from $i$ to $j$. |
| Shortest-path distance $d_{ij}$ | How far apart two regions are in the network, not in space. | Strong tracts count as short routes. | Edge length $1/w_{ij}$; $d_{ij}$ is the smallest total length over all paths ($\infty$ if none). |
| Node strength $k_i$ | How embedded a region is in the network. | A spared region whose tracts are cut loses strength: cortex that is present but disconnected. | $k_i=\sum_j w_{ij}$. |
| Global efficiency $E$ | How readily, in the graph sense, the network could integrate activity. | A whole-network summary; it can hide which tract was lost. | $E=\frac{1}{n(n-1)}\sum_{i\ne j}1/d_{ij}$; disconnected pairs contribute 0 (Latora and Marchiori 2001). |
| Hub | A highly connected, central region that supports integration (van den Heuvel and Sporns 2013). | Damage to hubs tends to disturb network organisation most (Aerts et al. 2016). | High strength or centrality. Not computed here: on 14 nodes of a complete graph it carries little information. |
| Spectral radius $\rho(W)$ | How strongly activity could reverberate through the network as a whole. | No clinical counterpart. | Largest eigenvalue modulus; for nonnegative symmetric $W$, the largest eigenvalue. The model is stable when $g\,\rho(W)<1$. |
| Disconnection, tract integrity $s_{ij}$ | Loss of a connection. The disconnection account holds that a function can fail when an intact region loses its connections. | The counterpart of grading a tract as intact, partly cut or cut. | $s_{ij}\in[0,1]$, computed from the density along the path; lesioned weight $s_{ij}w_{ij}$. |
| Diaschisis | Dysfunction in an intact region remote from a lesion, caused by lost connections (Fornito et al. 2015). | Remote effects of a tumor or a resection on regions it did not touch. | In this model: a correlation change between two regions whose own tract is intact. |
| Plasticity, reorganisation | Change in function and connections after damage. | Part of why patients may recover after resection in eloquent areas; white-matter damage limits it (Duffau 2014). | Absent: $W$ changes only through damage. |
| Coupling $g$ | How strongly regions drive each other in the model. | No clinical counterpart; not measured. | Scalar in $[0,1)$ multiplying $W$; at $g=0$ the regions are independent. |
| Model activity $x_i$ | A stand-in for slow regional activity; not BOLD and not a firing rate. | Not a recording. | Dimensionless state; time in units of a node time constant $\tau$. |
| Lyapunov equation | The condition that the activity's variability is in steady state. | — | $A\Sigma+\Sigma A^\top+\sigma^2I=0$ with $A=gW-I$; its solution is the stationary covariance $\Sigma$. |
| Null model, control | "What would the result look like if the hypothesis were false?" | Like a matched control group: the same amount of damage in a different place. | `matched_random_integrity` (same integrities on random tracts) and `callosum_sparing` (same density, callosum uninvolved). |
| Phantom | A constructed slice; region names are analogies. | Not a patient and not an atlas. | The `CoronalPhantom` of notebook 02 on a 1 mm grid. |

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

try:
    import google.colab  # noqa: F401

    IN_COLAB = True
except ModuleNotFoundError:
    IN_COLAB = False


def repository_root():
    candidates = (Path.cwd(), *Path.cwd().parents)
    return next((p for p in candidates if (p / "pyproject.toml").is_file()), None)


if IN_COLAB and repository_root() is None:
    checkout = Path("/content/AI-assisted-glioma-research")
    if not checkout.is_dir():
        url = "https://github.com/arvidl/AI-assisted-glioma-research"
        subprocess.run(["git", "clone", "--depth", "1", url, str(checkout)], check=True)
    os.chdir(checkout)
    sys.path.insert(0, str(Path.cwd() / "src"))
if repository_root() is None:
    raise RuntimeError("Open this notebook in a checkout of the repository or in Google Colab.")

In [ ]:
from pathlib import Path

import numpy as np
from IPython.display import Image, display

from glioma_ai.network_figures import write_network_experiment

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file())
OUTPUT = ROOT / "outputs/notebook11"
record = write_network_experiment(OUTPUT)
print("Nodes:", ", ".join(record["network"]["nodes"]))
print("Edge rule:", record["network"]["edge_rule"])
print("Damage rule:", record["network"]["damage_rule"])

## 1. Graphs and networks: a primer for this notebook

**Aim.** The rest of the notebook uses a handful of ideas from graph theory. This section defines
them, says where they come from, and explains why neuroscience, clinical neuroscience and
neuro-oncology adopted them. Readers who already know graph theory can skip to section 2 and come
back to the definitions when needed.

### 1.1 Graph theory and network science

Graph theory studies objects made of *nodes* (vertices) and *edges* (links between pairs of nodes).
It is usually dated to Euler's 1736 solution of the Königsberg bridge problem. Euler showed that
whether a walk can cross each of the city's bridges exactly once depends only on how the land masses
are connected, not on distances or shapes. The idea that some questions about a system can be
answered from its pattern of connections alone is the one this notebook uses.

In the twentieth century graph theory became a tool of the social sciences, physics and computing.
[Katz (1953)](https://doi.org/10.1007/BF02289026) ranked members of a social network by counting the walks that reach them,
each walk weighted down by its length; the same mathematical object reappears in section 3 as the
covariance of the model. *Network science* grew out of graph theory around the turn of the century,
when [Watts and Strogatz (1998)](https://doi.org/10.1038/30918) showed that a few random long-range links give a highly
clustered network short paths between any two nodes, which they called a "small world".
[Newman (2003)](https://doi.org/10.1137/S003614450342480) reviewed the field.

### 1.2 Definitions used in this notebook

A graph $G=(V,E)$ has $n$ nodes. Its *adjacency matrix* $A$ has $A_{ij}=1$ if nodes $i$ and $j$ are
joined by an edge and $0$ otherwise. In a *weighted* graph each edge carries a number $w_{ij}\ge0$,
collected in a weight matrix $W$. Here the graph is *undirected* ($W=W^\top$) and has no self-loops
($w_{ii}=0$).

- **Degree, strength and density.** The degree of node $i$ is its number of edges; its *strength*
  is $k_i=\sum_j w_{ij}$. The *density* is the fraction of the $n(n-1)/2$ possible pairs that have
  an edge.
- **Walks and paths.** A *walk* of length $k$ is a sequence of $k$ edges from $i$ to $j$; it may
  revisit nodes. A *path* is a walk that does not. The key identity is
  $$(W^k)_{ij}=\sum_{\text{walks } i=v_0\to v_1\to\dots\to v_k=j}\ w_{v_0v_1}w_{v_1v_2}\cdots w_{v_{k-1}v_k},$$
  so for a binary graph $(A^k)_{ij}$ counts the walks of length $k$ from $i$ to $j$.
- **Distance.** To turn weights into distances, a strong connection is treated as a short one: an
  edge has length $1/w_{ij}$, and $d_{ij}$ is the smallest total length over all paths from $i$ to
  $j$ ($d_{ij}=\infty$ if there is none).
- **Global efficiency.** $E=\frac{1}{n(n-1)}\sum_{i\ne j}1/d_{ij}$
  ([Latora and Marchiori 2001](https://doi.org/10.1103/PhysRevLett.87.198701)). Averaging inverse distances means that a disconnected
  pair contributes $0$ instead of making the average infinite, so $E$ stays defined when a lesion
  splits the network. A complete binary graph has $E=1$, and $E$ falls as paths lengthen.
- **Spectrum.** A symmetric $W$ has real eigenvalues. Its *spectral radius* $\rho(W)$ is the largest
  eigenvalue modulus. For a matrix with nonnegative entries, the Perron–Frobenius theorem says that
  $\rho(W)$ is itself an eigenvalue, with an eigenvector whose entries are nonnegative, and that
  $\rho$ cannot increase when entries decrease: if $0\le W'\le W$ entry by entry, then
  $\rho(W')\le\rho(W)$. A lesion only lowers weights, which is why, in section 3, a lesioned model
  stays stable.
- **Neumann series.** If $g\,\rho(W)<1$, then
  $$(I-gW)^{-1}=I+gW+g^2W^2+g^3W^3+\cdots.$$
  Read with the walk identity, entry $ij$ of this inverse adds up every walk from $i$ to $j$, each
  walk of length $k$ weighted by $g^k$. Katz's status index of a node sums these walk weights, from
  length 1 upwards, over all walks that reach it. The matrix itself is, up to a constant, the
  covariance of the model in section 3.

Some well-known measures are deliberately not used: the clustering coefficient, modularity,
small-worldness, hubs and rich clubs ([Rubinov and Sporns 2010](https://doi.org/10.1016/j.neuroimage.2009.10.003) review them and how to
interpret them). Section 2 shows that the phantom graph is complete, with an edge between every pair,
so binary measures are trivial there, and 14 nodes are too few for community or hub statistics to
mean much.

### 1.3 From graphs to brain networks

Applying graph theory to the brain means deciding what a node and an edge are.
[Sporns, Tononi and Kötter (2005)](https://doi.org/10.1371/journal.pcbi.0010042) proposed the *connectome*, a connection matrix of
the human brain. [Bullmore and Sporns (2009)](https://doi.org/10.1038/nrn2575) reviewed graph analysis of *structural*
networks, whose edges come from tract tracing or diffusion tractography, and of *functional*
networks, whose edges are correlations or synchronisation between recorded signals.
[Bassett and Sporns (2017)](https://doi.org/10.1038/nn.4502) describe network neuroscience as a field, and
[Fornito, Zalesky and Bullmore (2016)](https://doi.org/10.1016/C2012-0-06036-X) is a textbook.

Two cautions apply to everything below. First, network measures depend on how nodes are defined.
[Zalesky et al. (2010)](https://doi.org/10.1016/j.neuroimage.2009.12.027) found that parameters such as small-worldness and efficiency
varied considerably with the number of nodes in a parcellation, so values can only be compared at
the same nodal scale. Second, structural and functional networks on the same nodes are different
objects. One records wiring and the other a statistical dependence of activity; section 3 makes
their relation explicit for one model.

### 1.4 Networks in clinical neuroscience and neuro-oncology

[Fornito, Zalesky and Breakspear (2015)](https://doi.org/10.1038/nrn3901) review how network topology shapes the
brain's response to damage. They describe maladaptive processes, such as diaschisis, and resources
and processes that allow adaptation, such as degeneracy, reserve and compensation.
[Stam (2014)](https://doi.org/10.1038/nrn3801) reviews network science across neurological disorders and points to the
overload and failure of hubs as a possible common pathway. Hubs, the highly connected regions that
support integration, are also points of vulnerability ([van den Heuvel and Sporns 2013](https://doi.org/10.1016/j.tics.2013.09.012)).

In brain tumors, [Bartolomei et al. (2006)](https://doi.org/10.1016/j.clinph.2006.05.018) applied graph analysis to resting MEG in
17 patients and 15 controls. Functional connectivity was altered in a way that was not focal, and
the small-world architecture of the network changed. [Derks, Reijneveld and Douw (2014)](https://doi.org/10.1097/CCO.0000000000000126)
reviewed evidence that cognitive deficits in brain-tumor patients are often general (attention,
working memory, processing speed, executive function) rather than tied to tumor location, that they
are reflected in whole-brain network disturbances, and that network changes after resection go hand
in hand with cognitive outcome. [Herbet and Duffau (2020)](https://doi.org/10.1152/physrev.00033.2019) propose a
meta-networking theory, in which cognition arises from interacting distributed networks; the theory
also accounts for compensation after lesions. [Aerts et al. (2016)](https://doi.org/10.1093/brain/aww194) review how robust
brain networks are to tumors, stroke and traumatic injury.

**What each track takes from this section.**
- *Cognitive neuroscience.* The walk identity is the formal version of "influence through indirect
  routes". It is why, in section 3, two regions correlate without a direct edge, and why, in
  section 4, a lesion can change the correlation of two regions whose own connection is intact.
  That is a model form of diaschisis.
- *Neuro-oncology and clinical.* A tumor or a cavity damages edges; it need not remove any node.
  Node strength measures how far a spared region has been disconnected, and efficiency summarises
  the whole network. Neither says which function is lost.
- *Data analysis and modelling.* Three facts carry the rest: the weights are nonnegative and
  symmetric; Perron–Frobenius makes $\rho$ monotone under damage; and the Neumann series ties
  stability, the exact covariance and walks together. The cell below checks them on the smallest
  graph that has an indirect route.

**A three-node example.** Regions A, B and C form a chain: A–B and B–C are connected, A–C is not.
With unit weights divided by the spectral radius $\sqrt2$, the cell computes the model correlation
of section 3, the share of the A–C covariance carried by walks up to each length, and the global
efficiency.

In [ ]:
from glioma_ai.network import global_efficiency, linear_fc, walk_shares

chain = np.array([[0, 1, 0], [1, 0, 1], [0, 1, 0]]) / np.sqrt(2)  # A–B–C; spectral radius 1
_, toy_fc = linear_fc(chain, coupling=0.8)
ends = np.zeros((3, 3), dtype=bool)
ends[0, 2] = ends[2, 0] = True
print("Edges A–B and B–C; no A–C edge")
print(f"Model correlation at g = 0.8:  A–B {toy_fc[0, 1]:.3f}   A–C {toy_fc[0, 2]:.3f}")
shares = walk_shares(chain, 0.8, max_length=6, pairs=ends)
print("Share of A–C covariance from walks of length ≤ k, k = 0…6:", np.round(shares, 3))
print(f"Global efficiency {global_efficiency(chain):.3f} (1/d with edge length 1/w)")

**Reading the output.** A and C have no edge, yet their model correlation is positive: every route
between them passes through B. For this chain the exact value is $a^2/(1-a^2)$ with $a=g/\sqrt2$
(warm-up exercise 4). The walk shares rise only at even lengths, because the chain is
*bipartite*: every step alternates between B and an end, so a walk from A to C always has even
length. Global efficiency is below the value of a complete graph because A and C are reached only
through B.

## 2. The phantom network: nodes, edges and weights

**Aim.** Build a structural network on the coronal phantom of notebook 02 that is simple enough to
check by eye but keeps the anatomical property that matters most here: the two hemispheres are
joined only through the corpus callosum.

**Construction.**
- *Nodes.* 14 grey-matter regions, seven per hemisphere: the medial wall above the callosum, five
  angular sectors of the cortical ribbon, and the thalamus.
- *Edges.* For each pair of regions, the shortest 8-connected path through white matter on the
  1 mm grid, found with Dijkstra's algorithm. A path enters and leaves grey matter only at its ends.
- *Weights.* $w_{ij}=\exp(-L_{ij}/\lambda)$, with $L_{ij}$ the path length and decay length
  $\lambda=40$ mm.

Symbols:
- $i,j$: regions (nodes); $L_{ij}$ (mm): length of the white-matter path between them.
- $w_{ij}$ (dimensionless, 0–1): healthy structural weight; $w_{ii}=0$.
- Node strength $k_i=\sum_j w_{ij}$; interhemispheric weight
  $\sum_{i<j,\ \text{different sides}} w_{ij}$.
- Global efficiency $E$, with edge length $1/w_{ij}$ (section 1.2).

**Cognitive neuroscience track.** Seven regions per side stand in for the medial wall, cortical
sectors and thalamus at one coronal level. A name such as "superior parietal" is an analogy, not a
Desikan–Killiany label. Because the hemispheres meet only in the corpus callosum, every left–right
path crosses it. Any function that depends on a tract leaving this plane, such as frontoparietal
association fibres, cannot be represented.

**Neuro-oncology and clinical track.** A tract here is a shortest path through white matter on a
1 mm grid. It cannot leave the plane, and it hugs boundaries because a shortest path always takes
the tightest corner. It is a device for making a lost connection visible, not a diffusion
tractogram and not a surgical target.

**Data-analysis and modelling track.** The weight rule has the form of the exponential distance
rule reported for macaque tract tracing ([Ercsey-Ravasz et al. 2013](https://doi.org/10.1016/j.neuron.2013.07.036)). The 40 mm is a
teaching choice, not a fit; exercise 1 of the modelling track varies it. The first cell lists each
region's size and centroid on the phantom grid, and the second summarises the graph with the
measures of section 1.

In [ ]:
from glioma_ai.growth_interactive import _cached_tissue
from glioma_ai.network import phantom_network
from glioma_ai.network_figures import DEFAULT_GROWTH

phantom, runs, _ = _cached_tissue(*DEFAULT_GROWTH)
network = phantom_network(phantom)
print(f"{'node':<12}{'analogue':<36}{'cells (mm²)':>12}{'centroid x, y (mm)':>22}")
for k, (name, text) in enumerate(zip(network.names, network.descriptions, strict=True)):
    x, y = network.centroid_mm[k]
    print(f"{name:<12}{text:<36}{(network.labels == k).sum():>12}{x:>12.1f}, {y:6.1f}")
cross = network.interhemispheric
off = ~np.eye(network.size, dtype=bool)
within, across = network.length_mm[off & ~cross], network.length_mm[cross]
print(
    f"\nPath length within hemispheres {within.min():.0f}–{within.max():.0f} mm, "
    f"across {across.min():.0f}–{across.max():.0f} mm (decay length λ = {network.decay_mm:g} mm)"
)

In [ ]:
from glioma_ai.network import global_efficiency, spectral_radius

upper = network.weight[np.triu_indices(network.size, 1)]
strength = network.weight.sum(axis=1)
print(
    f"Nodes {network.size}; region pairs {upper.size}; pairs joined by a path "
    f"{np.count_nonzero(upper)} (binary density {np.count_nonzero(upper) / upper.size:.2f})"
)
print(
    f"Weights {upper.min():.3f}–{upper.max():.3f}; "
    f"node strength {strength.min():.2f}–{strength.max():.2f}"
)
print(
    f"Spectral radius ρ(W) {spectral_radius(network.weight):.3f}; "
    f"global efficiency {global_efficiency(network.weight):.3f}"
)

**Reading the output.** Every pair of regions is joined by a white-matter path, so as a binary graph
the phantom is complete (density 1): degree carries no information, and all of the structure is in
the weights, which span more than an order of magnitude. This is why the notebook uses weighted
measures only. Paths across the midline are longer than paths within a hemisphere (the ranges are
printed above), so interhemispheric weights are smaller. With $\lambda=40$ mm, a 100 mm path keeps
about 8% of the weight of a very short one. Real tractography connectomes are sparse, so the
density of this graph is a property of the construction, not of brains.

## 3. From structure to function: a linear model of correlated activity

**Aim.** Structural connectivity says which regions are wired together. What can be measured in
patients at rest is functional connectivity: which regions' activity rises and falls together. To
ask how a tumor's damage to the wiring changes function, we need a rule that turns $W$ into
correlations. This section introduces the simplest such rule that has an exact solution, shows what
it implies through the walks of section 1, and checks the numerics before the model is used in
Figures 1 and 2.

**The model.** Each region carries a dimensionless activity $x_i(t)$, with time in units of a node
time constant $\tau$:

$$\frac{d\mathbf x}{dt}=-\mathbf x+gW\mathbf x+\sigma\boldsymbol\xi(t),$$

- $-\mathbf x$: each region relaxes back to rest on its own.
- $gW\mathbf x$: each region is driven by its neighbours in proportion to the structural weights.
  $W$ is the weight matrix divided by the healthy network's largest eigenvalue, so the same scale
  is used before and after a lesion, and $g$ is a single global coupling.
- $\sigma\boldsymbol\xi$: independent white noise driving every region; without it the activity
  would decay to zero and there would be nothing to correlate.

**The exact solution.** With $A=gW-I$, the stationary covariance $\Sigma$ solves the Lyapunov
equation $A\Sigma+\Sigma A^\top+\sigma^2I=0$. When $W$ is symmetric and $g\,\rho(W)<1$, the solution
is $\Sigma=\tfrac{\sigma^2}{2}(I-gW)^{-1}$; substituting it gives
$A\Sigma+\Sigma A=-\sigma^2 I$ directly. Model functional connectivity is the correlation
$\Sigma_{ij}/\sqrt{\Sigma_{ii}\Sigma_{jj}}$, so $\sigma$ cancels.

**Why indirect routes matter.** By the Neumann series of section 1.2,
$$\Sigma=\tfrac{\sigma^2}{2}\big(I+gW+g^2W^2+\cdots\big),$$
so the covariance of two regions adds up every walk between them, a walk of length $k$ weighted by
$g^k$. At $g=0$ only the identity remains: the covariance is diagonal and the regions are
independent, whatever the wiring. As $g$ approaches $1/\rho(W)$, long walks are barely damped and
correlations spread across the network. The condition $g\,\rho(W)<1$ that makes the model stable is
the same condition that makes the series converge. Linear models of this kind have been used to
show how network architecture shapes spontaneous activity patterns
([Fernández Galán 2008](https://doi.org/10.1371/journal.pone.0002148)); [Deco, Jirsa and McIntosh (2011)](https://doi.org/10.1038/nrn2961) review dynamical models
of resting-state activity more broadly. This activity is not a BOLD signal: there is no
haemodynamic model, and the correlations are not fitted to any recording.

The cell below splits the healthy phantom's covariance by walk length.

In [ ]:
from glioma_ai.network import spectral_radius, walk_shares

scaled = network.weight / spectral_radius(network.weight)
for label, mask in (("all pairs", None), ("interhemispheric pairs", network.interhemispheric)):
    shares = walk_shares(scaled, 0.8, max_length=5, pairs=mask)
    print(
        f"g = 0.8, {label:<23} share of covariance from walks of length ≤ 1, 2, 3, 5: "
        + ", ".join(f"{shares[k]:.2f}" for k in (1, 2, 3, 5))
    )

**Reading the output.** At $g=0.8$, direct edges (walks of length 1) carry only a minority of the
model's covariance, and less for interhemispheric pairs, whose direct edges are weak. Most of the
covariance comes from longer walks through other regions. A lesion can therefore change the
correlation of two regions without touching the edge between them, which is what section 4 shows.

**Cognitive neuroscience track.** The correlation is a claim about routes. Direct weights matter,
and so do walks of length two and more. It is a claim about this equation, not about measured
resting-state networks. [Honey et al. (2009)](https://doi.org/10.1073/pnas.0811168106) compared a model with measured FC; here
there is no measurement to compare with.

**Neuro-oncology and clinical track.** Dividing by the healthy network's eigenvalue keeps the
preoperative and postoperative networks on one scale, so a change in correlation is not an artefact
of renormalising each network by itself. A real postoperative scan also has oedema, retraction and,
later, plasticity. None of these enters $W$.

**Data-analysis and modelling track.** Because $W$ is symmetric, the covariance is a matrix
inverse, not a simulation. A lesion only removes weight, so by Perron–Frobenius the largest
eigenvalue cannot rise, and a stable healthy model stays stable. The cell below checks the numerical
scheme against the exact covariance: first the bias from the time step, then the sampling error of
a finite run.

In [ ]:
controls = record["controls"]
print(f"Lyapunov residual (max |AΣ + ΣAᵀ + I|): {controls['lyapunov_residual_max']:.1e}")
print("Time-step bias of Euler–Maruyama (exact discrete vs continuous correlation, no sampling):")
for row in controls["step_bias"]:
    print(f"  dt = {row['dt_tau']:<6g} τ   max |Δ correlation| = {row['max_abs_difference']:.4f}")
print("Sampling error (simulated vs exact discrete correlation, dt = 0.1 τ):")
for row in controls["sampling"]:
    print(f"  duration {row['duration_tau']:>7g} τ   RMS error = {row['rms_error']:.4f}")

**Why these checks, and what they show.** The figures use the exact formula, so they do not depend
on simulation. The simulation is checked anyway, because notebook 12 and any nonlinear extension
would rely on it, and a simulation can be wrong in two separable ways.
- *Lyapunov residual.* The formula solves the equation to rounding error.
- *Time-step bias.* The Euler–Maruyama scheme with step $\Delta t$ has its own exact stationary
  covariance. Comparing that with the continuous one isolates the discretisation error from
  randomness. The error halves each time the step halves, as expected for a first-order scheme.
- *Sampling error.* A run of duration $T$ estimates correlations with an error that should fall
  roughly as $T^{-1/2}$. Each row is a single random realisation, so individual rows scatter about
  that trend; the test requires only that the error falls with longer runs.

These checks verify that the code solves the stated equations. They do not show that the equations
describe a brain.

## 4. Lesioning the network: tumor infiltration as edge damage (Figure 1)

**Aim.** Ask how a tumor that grows along white matter changes the network of section 2 and the
model functional connectivity of section 3. In particular: can a tumor confined to one hemisphere
weaken the connections between the hemispheres?

**Task.** At each day of notebook 02's growth runs, compute each tract's integrity from the tumor
density along its path, multiply the weights by it, and recompute the interhemispheric weight,
global efficiency and model FC. The damage rule is
$s_{ij}=\exp\!\big(-\tfrac{\Delta s}{\ell_d}\sum_{k\in\text{path}}c_k\big)$ with $\ell_d=10$ mm:
a tract that runs 10 mm through fully infiltrated tissue ($c=1$) keeps $e^{-1}\approx37\%$ of its
weight.

**Controls.** The three growth scenarios of notebook 02 differ only in how tissue guides the tumor:
uniform tissue, a grey/white contrast, and the contrast plus fibre anisotropy. A fourth curve keeps
the tissue-guided density but sets it to zero in the corpus callosum, which separates the loss due
to callosal infiltration from the loss due to infiltration elsewhere along the same tracts.

**What to look for.** In panel A, which tracts lose integrity first. In panel D, whether the
uniform-tissue curve falls even though that tumor stays in one hemisphere, and how far apart the
solid and dashed red curves are. In panel C, whether correlations change only for damaged pairs.

In [ ]:
display(Image(filename=str(OUTPUT / "network_lesion.png")))
print(
    f"Day {record['day']:g}; healthy mean interhemispheric model FC "
    f"{record['healthy_mean_interhemispheric_fc']:.3f}"
)
for name, values in record["scenarios_at_day"].items():
    print(
        f"{name:<46} interhemispheric weight {values['interhemispheric_weight_percent']:5.1f}%  "
        f"efficiency {values['global_efficiency_percent']:5.1f}%  "
        f"mean interhemispheric FC {values['mean_interhemispheric_fc']:.3f}"
    )

**Figure 1 | Synthetic tract disconnection by a tissue-guided tumor on a constructed
coronal phantom.** **A**, Density proxy $c$ of notebook 02's tissue-guided run (grey/white
contrast with fibre anisotropy) at day 180, ten days after $c\ge0.1$ first crosses the midline,
with the 91 tract proxies coloured by assumed integrity $s_{ij}$ (green intact, red cut) and drawn
wider for stronger healthy weights; black dots mark region centroids. **B**, Healthy weights
$w_{ij}=\exp(-L_{ij}/40\ \text{mm})$; lines separate the patient-right (top/left) and
patient-left regions. **C**, Change in model functional connectivity (correlation) between the
lesioned and healthy networks. **D**, Interhemispheric structural weight over the year for the
three growth scenarios of notebook 02 (grey: uniform tissue; blue: grey/white contrast; red:
contrast plus fibre anisotropy) and for the tissue-guided tumor with the corpus callosum assumed
uninvolved (dashed); the dotted line marks day 180. Radiological display: the patient's left is on
the viewer's right. All quantities are synthetic.

**Interpretation.** Four observations follow from the construction, not from biology.
1. *A tumor that does not cross can still weaken crossing connections.* In notebook 02 the
   uniform-tissue tumor never reaches $c\ge0.1$ in the right hemisphere within the year, yet the printed
   interhemispheric weight at day 180 is below healthy. Every interhemispheric tract from a left
   region runs through left white matter on its way to the callosum, and some of that white matter
   is infiltrated. In graph terms, what damages an edge is where the edge runs, not where its end
   nodes are.
2. *Callosal involvement adds a further loss.* The gap between the solid and dashed red curves in
   panel D is the extra loss that this damage rule attributes to infiltration of the callosum
   itself.
3. *Function changes beyond the damaged edges.* In panel C, correlation falls most for pairs whose
   tracts are damaged, but some pairs with intact tracts change as well. Section 3 explains why:
   much of their covariance runs along walks that cross damaged edges. This is the model form of
   diaschisis. [Honey et al. (2009)](https://doi.org/10.1073/pnas.0811168106) likewise found strong functional connections between
   regions without a direct structural connection, in both their model and their data.
4. *Efficiency falls less than interhemispheric weight.* Global efficiency averages over all pairs,
   most of them within one hemisphere, and shortest distances can route around a damaged edge. A
   whole-network summary can therefore look mild while a specific class of connections is badly
   damaged.

**Cognitive neuroscience track.** Panel C is the model's version of an indirect, remote effect. An
intact tract can still change its correlation because activity passes through other regions;
setting the coupling to zero removes every indirect route (cognitive exercise 1).
[Houskamp et al. (2025)](https://doi.org/10.3390/cancers17172775) report that, outside a few critical regions, high-grade gliomas
impair cognition generally. Damaged long-range connections are one candidate reason, and
[Griffis et al. (2019)](https://doi.org/10.1016/j.celrep.2019.07.100) found in stroke that interhemispheric disconnection in particular
tracked widespread FC disruption. This figure does not test either report.

**Neuro-oncology and clinical track.** The uniform-tissue curve is the bedside point of this
construction: laterality alone does not settle which connections a tumor affects. A tumor that stays
in one hemisphere can still reduce connections to the other, because those connections pass through
it on the way to the corpus callosum. The dashed red curve is the control that asks how much of the
loss is due to the callosum itself, the question that motivates callosal tractography in studies
such as [Wang et al. (2026)](https://doi.org/10.3171/2026.4.JNS26181).

**Data-analysis and modelling track.** All four observations are consequences of the path geometry
and of the rule for $s_{ij}$, not estimates. The size of the gap between the solid and dashed red
curves belongs to the attenuation rule with $\ell_d=10$ mm. The modelling exercises ask whether
another decay length keeps the order of the three growth scenarios, and what the harsher `max` rule
does to the callosum-sparing control.

## 5. A constructed resection: removing tumor also removes connections (Figure 2)

**Aim.** Ask what a resection does to the same network: which regions lose connections even though
the cavity spares them, and how the model FC changes from before to after surgery.

**Task.** At day 120, remove left-hemisphere white matter where the density proxy is at least 0.3,
sparing the corpus callosum and all grey-matter regions. A tract that crosses the cavity is cut
($s_{ij}=0$); residual infiltration outside the cavity still weakens tracts as in section 4. Node
strength and model FC are then compared across three networks: healthy, preoperative (tumor only)
and postoperative (cavity plus residual tumor).

**What to look for.** In panel B, whether spared cortical regions lose strength. In panel C, which
correlations change after surgery. Panel D is a template for a cognitive comparison, not a result.

In [ ]:
display(Image(filename=str(OUTPUT / "network_resection.png")))
resection = record["resection"]
print(
    f"Resection at day {resection['day']:g}: cavity {resection['cavity_area_mm2']:g} mm² "
    f"(c ≥ {resection['threshold']:g}; {resection['rule']})"
)
print("Interhemispheric weight (% of healthy):", resection["interhemispheric_weight_percent"])
print("\nAssumed (unsourced) CANTAB regions, left-hemisphere node strength (% of healthy):")
for domain in record["resection"]["assumed_domain_left_strength_percent"]["healthy"]:
    pre = resection["assumed_domain_left_strength_percent"]["pre-operative"][domain]
    post = resection["assumed_domain_left_strength_percent"]["post-operative"][domain]
    text = "not represented in this slice" if pre is None else f"pre {pre:5.1f}   post {post:5.1f}"
    print(f"  {domain:<26}{text}")

**Figure 2 | A constructed resection on the same synthetic network.** **A**, Cavity
(black) removed at day 120: left-hemisphere white matter with density proxy $c\ge0.3$, sparing
the corpus callosum and all grey-matter regions; tracts are coloured by post-operative integrity.
**B**, Node strength $k_i$ of each region for the healthy (grey), pre-operative (orange) and
post-operative (purple) networks. **C**, Change in model functional connectivity from before to
after resection. **D**, Left-hemisphere node strength, relative to healthy, averaged over the
regions an assumed and unsourced table assigns to four CANTAB measures; planning (SOC) has no
region in this slice. All quantities are synthetic.

**Interpretation.**
- *Spared cortex is disconnected.* Removing infiltrated white matter beneath intact cortex lowers
  that cortex's node strength: the regions are present, but their tracts are cut. This is the
  construction's version of the clinical observation that white-matter damage, more than cortical
  removal, limits recovery ([Duffau 2014](https://doi.org/10.1016/j.cortex.2013.08.005)). It is not a surgical outcome.
- *The postoperative network is not the healthy network minus a hole.* Residual infiltration
  outside the cavity still weakens tracts, so preoperative damage and surgical cuts add up.
- *Interhemispheric weight falls further after resection* (printed above), because left tracts on
  their way to the callosum cross the cavity.

### 5.1 What a cognitive comparison would require

Panel D averages left-hemisphere node strength over regions that an assumed, unsourced table
assigns to four CANTAB measures. Turning it into a test would need:
- a sourced mapping from each measure to regions and tracts, and a slice or volume that contains
  them (planning, SOC, needs frontal regions this slice lacks);
- subject-specific tractography before and after surgery;
- many patients with preoperative and postoperative scores, and a statistical model of reliable
  change. Notebook 10 shows why a single case cannot carry that weight: one patient's paired scores
  are descriptive at the released precision.

Plasticity, oedema resolution, retraction injury and functional reorganisation after surgery are
all absent.

**Cognitive neuroscience track.** Panel B is the dissociation the disconnection account predicts
inside this model: the region is present, and its strength still falls. Panel D cannot test the
account. [Herbet (2024)](https://doi.org/10.1093/brain/awae093) treats plasticity as part of the postoperative prediction, and
[Herbet and Duffau (2020)](https://doi.org/10.1152/physrev.00033.2019) argue that cognition depends on interacting networks
rather than single regions; this figure has neither a plasticity term nor a network-level cognitive
model to compare with.

**Neuro-oncology and clinical track.** The cavity rule is a teaching choice: left white matter with
$c\ge0.3$, callosum and grey matter spared, at day 120. Lowering the threshold removes more of the
density proxy and cuts more tracts. That trade-off is why the explorer below exists; it is not a
recommendation. [Noll et al. (2025)](https://doi.org/10.1093/noajnl/vdaf192) related postoperative connectome change to the help
patients needed in daily activities. Panel D's bars are not that outcome, and they are not
Karnofsky scores.

**Data-analysis and modelling track.** Compare panels B and C with the matched-random control
(modelling exercise 3). If shuffling the same integrity values across tracts reproduces the change
in FC, the loss is not specific to where the tumor sits. Section 6 describes the analogous question
in notebook 12, with one patient's streamlines and a tumor mask.

### 5.2 Explore the network

The sliders set the observation (or surgery) day, the cavity threshold and the coupling $g$; the
checkbox switches between Figure 1 (lesion only) and Figure 2 (resection at the chosen day).
▶ plays the year in 5-day steps. Things to try:
- Play the year without resection. When does interhemispheric weight start to fall, and which
  tracts go first?
- Resect at day 60, 120 and 180 with the same threshold. How does the cost to intact cortex grow?
- Lower the threshold from 0.5 to 0.1. A larger cavity removes more tumor but cuts more tracts.
- Set $g=0$: every region becomes independent and model FC vanishes, whatever the structure.
  Raise $g$ towards 0.95: longer walks dominate and correlations spread.

**By track.** *Cognitive neuroscience:* at $g=0$, which panel of Figure 1 has to go flat, and why?
*Clinical:* for one threshold, write down which tracts you would be unwilling to cut, and whether
this phantom even contains them. *Modelling:* confirm that the preoperative and postoperative
correlations stay finite as $g$ approaches 1 from below, as the stability condition requires.

In [ ]:
from glioma_ai.network_figures import network_explorer

network_explorer()

## 6. What carries over to a patient connectome (notebook 12)

Notebook 12 asks whether the chain of this notebook, from the tumor through lost connections to
model functional connectivity, still describes one patient's measurements. The table maps each
element of the phantom to its counterpart there.

| Element | This notebook (phantom) | Notebook 12 (sub-PAT16) |
| --- | --- | --- |
| Nodes | 14 constructed regions on one coronal slice | 82 Desikan–Killiany cortical and subcortical regions |
| Edges and weights | Shortest white-matter paths; $\exp(-L/\lambda)$ | Streamline weights from diffusion tractography, compressed with $\log(1+\text{weight})$ |
| Lesion | Integrity from a synthetic density along each path | The share of each connection carried by streamlines through a tumor mask |
| Resection | A threshold on the density, callosum and grey matter spared | Every streamline that touches the preoperative tumor mask is deleted; the mask is not the postoperative cavity |
| Dynamics | Linear model, $g=0.8$, divided by the healthy spectral radius | The same model, divided by the preoperative spectral radius |
| Functional connectivity | Exact model correlation only | Model correlation compared with measured resting-state FC |
| Controls | Callosum sparing; matched random integrities | Tumor-crossing shares moved to random connected pairs; no callosum-sparing control, because the mask is wherever the tumor was |
| Cognitive link | Unsourced template (Figure 2D) | The patient's CANTAB scores beside the matrices; still no fitted link |

**The data.** sub-PAT16 is one participant in the Brain Tumor Connectomics release of Aerts,
Marinazzo and colleagues: the preoperative session is
[OpenNeuro ds001226 v5.0.0](https://doi.org/10.18112/openneuro.ds001226.v5.0.0) and the
postoperative session [ds002080 v4.0.0](https://doi.org/10.18112/openneuro.ds002080.v4.0.0), both
CC0. Notebook 10 uses the same datasets' participant tables at the later point releases v5.0.1 and
v4.0.1. The postoperative scan is about six months after surgery; notebook 12 records that interval
and does not use it as a variable. Notebook 12 does not read the images, tractograms or
demographics, which stay outside Git. It reads small tracked matrices in `data/derived/sub-PAT16/`,
which come with the clone, also in Colab. Their derivation reads the investigator's local upstream
files and stays in the archived development repository. The CANTAB cross-check needs a local OSF
copy and shows a notice without it.

**What has not been reviewed.** Notebook 12 states that the upstream quality control is still open:
surfaces, registration, eddy correction, tractography, the tumor mask (from a trained segmentation
model, not reviewed) and the parcellation after surgery. Its postoperative matrices are a later
measurement, not the model with a hole cut in it. A multi-patient study would also need a
prespecified MRI and cognitive protocol; [Berger et al. (2026)](https://doi.org/10.1111/ene.70708) report a Delphi consensus
on such acquisition for glioma. [Kong and Wei (2026)](https://doi.org/10.1002/nbm.70391) argue that a connectivity map should
not be used for one patient until the clinical state of the scan is recorded and the map's added
value is validated. Notebook 12 is a descriptive comparison, not that validation, and not a
clinical finding.

**What to look for, by track.**
- *Cognitive neuroscience.* Measured FC is a Pearson correlation of cleaned BOLD time series; the
  model's correlation is the solution of the Lyapunov equation. A resemblance between them is a
  comparison of two different quantities.
- *Neuro-oncology and clinical.* The case is the same sub-PAT16 as in notebook 10, so the CANTAB
  scores can be set beside the matrices. No age or sex is shown. Nothing in either notebook is a
  statement about what should have been resected.
- *Data-analysis and modelling.* Which controls of this notebook have an analogue there, and which
  do not (table above). Notebook 12 also checks its own reader by rebuilding the upstream all-label
  matrices and printing their edge correlation with MRtrix3 `tck2connectome`.

## 7. Exercises

Each exercise says what to use (pen and paper, the explorer, a printed output or a function) and
what to produce. Where possible, a *check* says how to tell whether you are right without an answer
key. Rebuilding the figures is deterministic and calls no language model. The figures
(`network_lesion`, `network_resection` as PNG and PDF) and `network_experiment.json`, with every
rule, parameter, control and summary value, are saved in the ignored folder `outputs/notebook11/`.
For the code exercises, the cells of section 2 define `phantom`, `runs` and `network`; the
tissue-guided run is `runs[list(runs)[-1]][0]`, with arrays `.days` and `.density`.

### 7.1 Warm-up for all tracks: graph theory
1. **Walks by hand** (pen and paper). For the chain A–B–C with unit weights, write $A$, $A^2$ and
   $A^3$. Which walks does each nonzero entry of $A^2$ count? Why is the A–C entry of every odd
   power zero? *Check:* the walk shares printed in section 1.
2. **Efficiency by hand** (pen and paper). Compute the global efficiency of the chain, first with
   unit weights and then with weights $1/\sqrt2$. *Check:* the second value is printed in
   section 1.
3. **Perron–Frobenius on two nodes** (pen and paper). For $W=\begin{pmatrix}0&w\\w&0\end{pmatrix}$
   with $w>0$, find the eigenvalues and $\rho(W)$. Show that lowering $w$ lowers $\rho$, and give
   the largest $g$ for which the model of section 3 is stable.
4. **An exact correlation** (pen and paper). For the chain with weights $1/\sqrt2$, invert
   $I-gW$ and show that the A–C correlation is $a^2/(1-a^2)$ with $a=g/\sqrt2$. *Check:* the
   printed A–C value at $g=0.8$.

### 7.2 Cognitive neuroscience track
1. **Indirect routes** (Figure 1C, explorer). Some intact pairs change correlation. Explain this
   from the walk expansion of section 3. Then set $g=0$ in the explorer and say what the same panel
   must become.
2. **A missing lobe.** Planning (SOC) has no region in panel D. Which tracts would a more anterior
   slice have to add before a planning score could be placed on this network, and which of those
   tracts can a coronal parietal phantom never contain?
3. **Spared cortex** (Figure 2A, 2B). State the disconnection prediction in one sentence, and the
   observation that would falsify it inside this model.
4. **What Honey et al. measured.** Their functional connectivity came from resting-state MRI. List
   two steps between this model's correlation and that measurement (haemodynamics is one).
5. **Diaschisis in the model** (code). Using `tract_integrity` and `linear_fc` at day 180, find a
   pair whose integrity stays above 0.99 but whose correlation changes most. Which damaged edges lie on the
   strongest short walks between them?

### 7.3 Neuro-oncology and clinical track
1. **The threshold** (explorer). Compare cavity thresholds 0.5, 0.3 and 0.1 at day 120. Describe the
   trade-off in two sentences a colleague could read. Do not call the result a surgical plan.
2. **Laterality is not enough** (Figure 1D). In three sentences for a tumor board, explain why a
   tumor confined to one hemisphere can weaken interhemispheric connections, and say what this
   construction leaves out.
3. **Omitted recovery.** Plasticity, oedema resolution, retraction injury and reorganisation are
   absent. For each, say whether adding it would be expected to raise or lower the postoperative
   drop in node strength, and what measurement would decide.
4. **From this figure to a review.** Tykocki and Rakasz (2026) review connectome-guided resection,
   tract preservation and postoperative cognition. List what this notebook would have to replace
   (phantom, damage rule, unsourced cognitive table, single constructed tumor) before a result here
   could be entered in such a review.
5. **Daily activities.** Noll et al. (2025) associated early postoperative connectome change with
   needing help in instrumental activities, and reported that those independence ratings were not
   associated with Karnofsky score, manual dexterity, tumor volume or extent of resection. Why do the
   bars in Figure 2D not speak to either finding?

### 7.4 Data-analysis and modelling track
1. **Decay length** (code). Rebuild the network with `phantom_network(phantom, decay_mm=...)` for
   two other values and check whether the order of the three growth scenarios in Figure 1D changes.
2. **A harsher rule** (code). With `tract_integrity(network, density, rule="max")`, compute the
   interhemispheric weight over the year with and without `callosum_sparing(phantom, density)`.
   The two curves nearly coincide. Explain why from the definition of the rule and from where the
   density is highest along each interhemispheric path.
3. **Specificity** (code). Compare the change in FC with `matched_random_integrity(network,
   integrity, seed=...)`, which places the same integrity values on random tracts, for several
   seeds. Is the loss specific to where the tumor is?
4. **Walk length** (code). With `walk_shares`, find the walk length needed to capture 90% of the
   healthy covariance at $g=0.5$, $0.8$ and $0.95$. Relate the answers to the geometric factor
   $g\,\rho(W)$ of the Neumann series.
5. **Order of the scheme.** The bias halves when the time step halves. What order is that, and
   what pattern should the sampling-error rows follow as the run lengthens?
6. **Eigenvalue.** Explain, from the sign of the weights, why a lesion cannot raise the largest
   eigenvalue. What would a negative weight do to that argument?
7. **The bridge.** A patient connectome replaces the phantom with diffusion MRI and tractography,
   and has to record registration, parcellation and quality control. Using the table in section 6,
   say which of this notebook's controls has an analogue in notebook 12 and which has none.

## 8. References

Recent work (2025–2026) is marked ★. Nothing below is a result of the phantom model.

How the references were checked. Every DOI was resolved in Crossref (the earlier entries on
14 October, the new ones on 15 October 2026), and the title, first author and year were compared
with each entry. Summaries of Salvalaggio, Cayuela, Douw, Herbet and Houskamp were
checked against PubMed abstracts for notebook 10, and the other ★ entries against Crossref and the
PubMed or publisher abstracts for this notebook (14 October 2026). On 15 October 2026 the following
were added and their PubMed abstracts read for the statements made here: Honey and Sporns 2008,
Griffis et al. 2019, Duffau 2014, Aerts et al. 2016, Abdelnour et al. 2014, Zalesky et al. 2010,
Fornito et al. 2015, Stam 2014, van den Heuvel and Sporns 2013, Bartolomei et al. 2006, Derks et al.
2014, Herbet and Duffau 2020 and Sporns et al. 2005. Katz 1953, Watts and Strogatz 1998, Newman
2003, Bullmore and Sporns 2009, Rubinov and Sporns 2010, Bassett and Sporns 2017, Deco et al. 2011
and the Fornito, Zalesky and Bullmore textbook are cited for their general subject only and were
checked for metadata; the textbook's Crossref record lists no authors. Euler's 1736 paper is
mentioned for history and not cited.

**Graph theory and network neuroscience (all tracks)**
- Katz L. A new status index derived from sociometric analysis. *Psychometrika* 1953;18:39–43. [doi:10.1007/BF02289026](https://doi.org/10.1007/BF02289026)
- Watts DJ, Strogatz SH. Collective dynamics of 'small-world' networks. *Nature* 1998;393:440–442. [doi:10.1038/30918](https://doi.org/10.1038/30918)
- Latora V, Marchiori M. Efficient behavior of small-world networks. *Phys Rev Lett* 2001;87:198701. [doi:10.1103/PhysRevLett.87.198701](https://doi.org/10.1103/PhysRevLett.87.198701)
- Newman MEJ. The structure and function of complex networks. *SIAM Rev* 2003;45:167–256. [doi:10.1137/S003614450342480](https://doi.org/10.1137/S003614450342480)
- Sporns O, Tononi G, Kötter R. The human connectome: a structural description of the human brain. *PLoS Comput Biol* 2005;1:e42. [doi:10.1371/journal.pcbi.0010042](https://doi.org/10.1371/journal.pcbi.0010042)
- Bullmore E, Sporns O. Complex brain networks: graph theoretical analysis of structural and functional systems. *Nat Rev Neurosci* 2009;10:186–198. [doi:10.1038/nrn2575](https://doi.org/10.1038/nrn2575)
- Rubinov M, Sporns O. Complex network measures of brain connectivity: uses and interpretations. *NeuroImage* 2010;52:1059–1069. [doi:10.1016/j.neuroimage.2009.10.003](https://doi.org/10.1016/j.neuroimage.2009.10.003)
- Zalesky A et al. Whole-brain anatomical networks: does the choice of nodes matter? *NeuroImage* 2010;50:970–983. [doi:10.1016/j.neuroimage.2009.12.027](https://doi.org/10.1016/j.neuroimage.2009.12.027)
- van den Heuvel MP, Sporns O. Network hubs in the human brain. *Trends Cogn Sci* 2013;17:683–696. [doi:10.1016/j.tics.2013.09.012](https://doi.org/10.1016/j.tics.2013.09.012)
- Fornito A, Zalesky A, Bullmore ET. *Fundamentals of Brain Network Analysis.* Academic Press (Elsevier); 2016. [doi:10.1016/C2012-0-06036-X](https://doi.org/10.1016/C2012-0-06036-X)
- Bassett DS, Sporns O. Network neuroscience. *Nat Neurosci* 2017;20:353–364. [doi:10.1038/nn.4502](https://doi.org/10.1038/nn.4502)

**Cognitive neuroscience track**
- Catani M, ffytche DH. The rises and falls of disconnection syndromes. *Brain* 2005;128:2224–2239. [doi:10.1093/brain/awh622](https://doi.org/10.1093/brain/awh622)
- Honey CJ, Sporns O. Dynamical consequences of lesions in cortical networks. *Hum Brain Mapp* 2008;29:802–809. [doi:10.1002/hbm.20579](https://doi.org/10.1002/hbm.20579)
- Honey CJ et al. Predicting human resting-state functional connectivity from structural connectivity. *Proc Natl Acad Sci USA* 2009;106:2035–2040. [doi:10.1073/pnas.0811168106](https://doi.org/10.1073/pnas.0811168106)
- Griffis JC et al. Structural disconnections explain brain network dysfunction after stroke. *Cell Rep* 2019;28:2527–2540.e9. [doi:10.1016/j.celrep.2019.07.100](https://doi.org/10.1016/j.celrep.2019.07.100)
- Herbet G, Duffau H. Revisiting the functional anatomy of the human brain: toward a meta-networking theory of cerebral functions. *Physiol Rev* 2020;100:1181–1228. [doi:10.1152/physrev.00033.2019](https://doi.org/10.1152/physrev.00033.2019)
- Herbet G. Predictors of cognition after glioma surgery: connectotomy, structure-function phenotype, plasticity. *Brain* 2024;147:2621–2635. [doi:10.1093/brain/awae093](https://doi.org/10.1093/brain/awae093)
- ★ Salvalaggio A et al. Insights from the brain connectome in patients with gliomas. *Curr Opin Oncol* 2025;37:595–602. [doi:10.1097/CCO.0000000000001190](https://doi.org/10.1097/CCO.0000000000001190)
- ★ Cayuela N et al. Mapping glioma's impact on cognition: insights from macrostructure, microstructure, and beyond. *Neuro-Oncol Adv* 2025;7:vdaf003. [doi:10.1093/noajnl/vdaf003](https://doi.org/10.1093/noajnl/vdaf003)
- ★ Houskamp E et al. Impact of high-grade glioma lesion location on preoperative neuropsychological deficits. *Cancers* 2025;17:2775. [doi:10.3390/cancers17172775](https://doi.org/10.3390/cancers17172775)
- ★ Douw L et al. Multiscale network perspectives on glioma: from tumour biology to symptoms, survival and treatment. *Nat Rev Neurol* 2026;22:73–89. [doi:10.1038/s41582-025-01171-x](https://doi.org/10.1038/s41582-025-01171-x)

**Neuro-oncology and clinical track**
- Bartolomei F et al. Disturbed functional connectivity in brain tumour patients: evaluation by graph analysis of synchronization matrices. *Clin Neurophysiol* 2006;117:2039–2049. [doi:10.1016/j.clinph.2006.05.018](https://doi.org/10.1016/j.clinph.2006.05.018)
- Duffau H. The huge plastic potential of adult brain and the role of connectomics: new insights provided by serial mappings in glioma surgery. *Cortex* 2014;58:325–337. [doi:10.1016/j.cortex.2013.08.005](https://doi.org/10.1016/j.cortex.2013.08.005)
- Derks J, Reijneveld JC, Douw L. Neural network alterations underlie cognitive deficits in brain tumor patients. *Curr Opin Oncol* 2014;26:627–633. [doi:10.1097/CCO.0000000000000126](https://doi.org/10.1097/CCO.0000000000000126)
- Stam CJ. Modern network science of neurological disorders. *Nat Rev Neurosci* 2014;15:683–695. [doi:10.1038/nrn3801](https://doi.org/10.1038/nrn3801)
- Fornito A, Zalesky A, Breakspear M. The connectomics of brain disorders. *Nat Rev Neurosci* 2015;16:159–172. [doi:10.1038/nrn3901](https://doi.org/10.1038/nrn3901)
- Aerts H, Fias W, Caeyenberghs K, Marinazzo D. Brain networks under attack: robustness properties and the impact of lesions. *Brain* 2016;139:3063–3083. [doi:10.1093/brain/aww194](https://doi.org/10.1093/brain/aww194)
- ★ Noll KR et al. Changes in functional connectivity are associated with functional independence in the early postoperative period following awake surgical resection of language-eloquent glioma. *Neuro-Oncol Adv* 2025;7:vdaf192. [doi:10.1093/noajnl/vdaf192](https://doi.org/10.1093/noajnl/vdaf192)
- ★ Tykocki T, Rakasz Ł. Connectome-guided glioma resection: a systematic review of white matter tract preservation and postoperative neurocognition. *Neurosurg Rev* 2026;49:191. [doi:10.1007/s10143-025-04081-0](https://doi.org/10.1007/s10143-025-04081-0)
- ★ Nasiri D et al. Brain tumor surgery affecting the left frontal aslant tract and clinical outcome: a study of 23 glioma patients. *Brain Spine* 2026;6:106254. [doi:10.1016/j.bas.2026.106254](https://doi.org/10.1016/j.bas.2026.106254)
- ★ Wang X et al. Use of structure-function coupling to assess compensatory motor cortex flexibility and predict postoperative motor outcomes in patients with motor area gliomas. *J Neurosurg* 2026:1–12. [doi:10.3171/2026.4.JNS26181](https://doi.org/10.3171/2026.4.JNS26181)
- ★ Berger T et al. Harmonized acquisition of connectome MRI and clinical data for stroke, Parkinson's disease and glioma: Delphi-derived protocol recommendations by the European Academy of Neurology. *Eur J Neurol* 2026;33:e70708. [doi:10.1111/ene.70708](https://doi.org/10.1111/ene.70708)
- ★ Kong SS, Wei LC. Clinical-state validation should precede patient-level use of glioma connectivity anomaly maps. *NMR Biomed* 2026;39:e70391. [doi:10.1002/nbm.70391](https://doi.org/10.1002/nbm.70391)

**Data-analysis and modelling track**
- Fernández Galán R. On how network architecture determines the dominant patterns of spontaneous neural activity. *PLoS ONE* 2008;3:e2148. [doi:10.1371/journal.pone.0002148](https://doi.org/10.1371/journal.pone.0002148)
- Alstott J et al. Modeling the impact of lesions in the human brain. *PLoS Comput Biol* 2009;5:e1000408. [doi:10.1371/journal.pcbi.1000408](https://doi.org/10.1371/journal.pcbi.1000408)
- Deco G, Jirsa VK, McIntosh AR. Emerging concepts for the dynamical organization of resting-state activity in the brain. *Nat Rev Neurosci* 2011;12:43–56. [doi:10.1038/nrn2961](https://doi.org/10.1038/nrn2961)
- Ercsey-Ravasz M et al. A predictive network model of cerebral cortical connectivity based on a distance rule. *Neuron* 2013;80:184–197. [doi:10.1016/j.neuron.2013.07.036](https://doi.org/10.1016/j.neuron.2013.07.036)
- Abdelnour F, Voss HU, Raj A. Network diffusion accurately models the relationship between structural and functional brain connectivity networks. *NeuroImage* 2014;90:335–347. [doi:10.1016/j.neuroimage.2013.12.039](https://doi.org/10.1016/j.neuroimage.2013.12.039)
- Aerts H et al. Modeling brain dynamics after tumor resection using The Virtual Brain. *NeuroImage* 2020;213:116738. [doi:10.1016/j.neuroimage.2020.116738](https://doi.org/10.1016/j.neuroimage.2020.116738)
- Aerts H et al. Pre- and post-surgery brain tumor multimodal MRI data optimized for large scale computational modelling. *Sci Data* 2022;9:676. [doi:10.1038/s41597-022-01806-4](https://doi.org/10.1038/s41597-022-01806-4)